# Customer Churn Prediction

**Telco Customer Churn — End-to-End Machine Learning Pipeline**

[![Python](https://img.shields.io/badge/Python-3.10+-blue.svg)](https://www.python.org/)
[![scikit-learn](https://img.shields.io/badge/scikit--learn-1.4-orange.svg)](https://scikit-learn.org/)
[![XGBoost](https://img.shields.io/badge/XGBoost-2.0-green.svg)](https://xgboost.readthedocs.io/)
[![License](https://img.shields.io/badge/License-MIT-lightgrey.svg)](https://opensource.org/licenses/MIT)

---

## Overview

This notebook implements a complete, production-style workflow for predicting customer churn in a
telecommunications company. It follows the standard structure used in top Kaggle kernels and reference
GitHub repositories for this problem (IBM Telco Customer Churn use case), covering:

1. Data acquisition and integrity checks
2. Exploratory Data Analysis (EDA)
3. Data cleaning and feature engineering
4. Preprocessing pipeline (encoding, scaling, class imbalance handling)
5. Model training and comparison (Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM)
6. Hyperparameter tuning
7. Model evaluation (ROC-AUC, PR-AUC, confusion matrix, classification report)
8. Feature importance and interpretability (SHAP)
9. Business insights and model persistence

## Dataset

**IBM Telco Customer Churn** — 7,043 customers, 21 features (demographics, account information,
subscribed services) with a binary `Churn` target.

- Kaggle: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- Source (raw CSV, IBM sample data): https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv


## 1. Environment Setup

In [ ]:
!pip install -q xgboost lightgbm imbalanced-learn shap

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve, auc
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap
import joblib

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

np.random.seed(RANDOM_STATE)

## 2. Data Loading

The dataset is pulled directly from the public IBM sample-data mirror on GitHub so the notebook is fully
reproducible without manual downloads.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_URL)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 3. Data Quality Checks

In [ ]:
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# TotalCharges is stored as object due to blank strings for new customers (tenure = 0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("\nMissing TotalCharges after coercion:", df["TotalCharges"].isnull().sum())
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

In [ ]:
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
df.drop(columns=["customerID"], inplace=True)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

assert df.isnull().sum().sum() == 0
df.shape

## 4. Exploratory Data Analysis

In [ ]:
churn_rate = df["Churn"].mean()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(x="Churn", data=df, ax=ax[0], hue="Churn", legend=False, palette=["#2E86AB", "#E63946"])
ax[0].set_xticklabels(["Retained", "Churned"])
ax[0].set_title("Class Distribution")
for p in ax[0].patches:
    ax[0].annotate(f"{p.get_height():,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom")

df["Churn"].value_counts().plot.pie(
    autopct="%1.1f%%", labels=["Retained", "Churned"], colors=["#2E86AB", "#E63946"],
    ax=ax[1], ylabel="", startangle=90
)
ax[1].set_title("Churn Proportion")
plt.tight_layout()
plt.show()

print(f"Overall churn rate: {churn_rate:.2%}")

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, col in enumerate(numeric_cols):
    sns.kdeplot(data=df, x=col, hue="Churn", fill=True, common_norm=False,
                palette={0: "#2E86AB", 1: "#E63946"}, alpha=0.5, ax=axes[i])
    axes[i].set_title(f"{col} Distribution by Churn")
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
categorical_cols.remove("Churn") if "Churn" in categorical_cols else None

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    churn_by_cat = df.groupby(col)["Churn"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=churn_by_cat.index, y=churn_by_cat.values, ax=axes[i], color="#457B9D")
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel("Churn Rate (%)")
    axes[i].tick_params(axis="x", rotation=40)

for j in range(len(categorical_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
corr = df[numeric_cols + ["Churn"]].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix (Numeric Features)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x="Contract", y="MonthlyCharges", hue="Churn", data=df, ax=ax, palette=["#2E86AB", "#E63946"])
ax.set_title("Monthly Charges by Contract Type and Churn")
ax.legend(title="Churn", labels=["Retained", "Churned"])
plt.tight_layout()
plt.show()

## 5. Feature Engineering

Derived features that consistently improve churn models on this dataset:
tenure buckets, average monthly spend, and a service-adoption count.

In [ ]:
def bucket_tenure(t):
    if t <= 12:
        return "0-1yr"
    elif t <= 24:
        return "1-2yr"
    elif t <= 48:
        return "2-4yr"
    else:
        return "4yr+"

df["TenureGroup"] = df["tenure"].apply(bucket_tenure)

service_cols = ["PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
                 "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

df["NumServices"] = df[service_cols].apply(
    lambda row: sum(v not in ["No", "No internet service", "No phone service"] for v in row), axis=1
)

df["AvgMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0, 1)

df[["tenure", "TenureGroup", "NumServices", "AvgMonthlySpend"]].head()

## 6. Train / Test Split and Preprocessing Pipeline

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(exclude="object").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}, Test churn rate: {y_test.mean():.2%}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

## 7. Model Training and Comparison

Class imbalance (~26.5% positive class) is handled with SMOTE oversampling inside the pipeline,
applied only to the training folds to avoid data leakage.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

results = []

for name, model in models.items():
    pipe = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", model),
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
results_df.style.background_gradient(cmap="Blues", subset=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
results_melt = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
sns.barplot(data=results_melt, x="Model", y="Score", hue="Metric", ax=ax)
ax.set_title("Model Comparison Across Metrics")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning

The best-performing model from the comparison above is tuned further with `GridSearchCV`
optimizing for ROC-AUC under 5-fold stratified cross-validation.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print(f"Best baseline model: {best_model_name}")

param_grid = {
    "classifier__n_estimators": [200, 400, 600],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
}

tuning_pipe = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    tuning_pipe, param_grid=param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print("Best CV ROC-AUC:", grid_search.best_score_)
print("Best params:", grid_search.best_params_)

best_pipe = grid_search.best_estimator_

## 9. Final Model Evaluation

In [ ]:
y_pred = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0],
            xticklabels=["Retained", "Churned"], yticklabels=["Retained", "Churned"])
ax[0].set_title("Confusion Matrix")
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_val = auc(fpr, tpr)
ax[1].plot(fpr, tpr, color="#E63946", lw=2, label=f"AUC = {roc_auc_val:.3f}")
ax[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
ax[1].set_title("ROC Curve")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].legend(loc="lower right")

precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc_val = auc(recall, precision)
ax[2].plot(recall, precision, color="#2E86AB", lw=2, label=f"AUC = {pr_auc_val:.3f}")
ax[2].set_title("Precision-Recall Curve")
ax[2].set_xlabel("Recall")
ax[2].set_ylabel("Precision")
ax[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

## 10. Feature Importance and Interpretability (SHAP)

In [ ]:
feature_names = (
    numeric_features
    + list(best_pipe.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_features))
)

xgb_model = best_pipe.named_steps["classifier"]
importances = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, color="#457B9D")
plt.title("Top 15 Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
X_test_transformed = best_pipe.named_steps["preprocessor"].transform(X_test)
X_test_transformed = pd.DataFrame(
    X_test_transformed.toarray() if hasattr(X_test_transformed, "toarray") else X_test_transformed,
    columns=feature_names
)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(shap_values, X_test_transformed, show=False, max_display=15)
plt.tight_layout()
plt.show()

## 11. Model Persistence

The full pipeline (preprocessing + SMOTE + tuned classifier) is serialized so it can be loaded
directly for batch or real-time inference without refitting.

In [ ]:
joblib.dump(best_pipe, "churn_model_pipeline.pkl")
print("Model saved to churn_model_pipeline.pkl")

def predict_churn(customer_df, pipeline=best_pipe, threshold=0.5):
    proba = pipeline.predict_proba(customer_df)[:, 1]
    label = (proba >= threshold).astype(int)
    return pd.DataFrame({"churn_probability": proba, "churn_prediction": label})

predict_churn(X_test.head(5))

## 12. Summary and Business Recommendations

**Key drivers of churn identified by the model:**

- **Contract type** — month-to-month customers churn at a substantially higher rate than one- or two-year contract holders.
- **Tenure** — churn risk is highest in the first 12 months of the customer lifecycle.
- **Internet service type** — fiber optic subscribers show elevated churn compared to DSL.
- **Add-on services** — lack of online security and tech support correlates with higher churn.
- **Payment method** — electronic check payers churn more than customers on automatic payment methods.

**Recommended actions:**

1. Target month-to-month customers within their first year with retention offers or discounted contract upgrades.
2. Bundle tech support and online security into fiber optic plans to increase stickiness.
3. Migrate electronic-check customers to automatic payment methods through incentives.
4. Deploy the trained pipeline as a monthly batch scoring job to flag high-risk accounts for the retention team.

---

**References**

- IBM Telco Customer Churn dataset — https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- IBM Cloud Pak for Data reference implementation — https://github.com/IBM/telco-customer-churn-on-icp4d
